In [14]:
import requests_cache
import json
import pandas as pd
from time import sleep

In [15]:
session = requests_cache.CachedSession(
    cache_name='Extra', use_cache_dir=True, expire_after=604800)
urls = {
    'PremierLeague': 'https://www.fotmob.com/api/leagues?id=47&ccode3=UK&season=2023%2F2024',
}

In [16]:
def all_matches_data(url):
    response = session.get(url)
    data = json.loads(response.content)
    ids = [box['id'] for box in data['matches']
           ['allMatches'] if box['status']['finished'] == True]
    return ids


ids = all_matches_data(urls['PremierLeague'])
len(ids)

285

In [17]:
def match_details(ids):
    all_teams = []
    
    for id in ids:
        base_data = {}
        home_data = {}
        away_data = {}
        response = session.get(f'https://www.fotmob.com/api/matchDetails?matchId={id}')
        data = json.loads(response.content)
        
        base_data['match_date'] = data['general']['matchTimeUTCDate'].split('T')[0]
        base_data['league_name'] = data['general']['leagueName']
        base_data['matchid'] = data['general']['matchId']
        base_data['home_team'] = data['general']['homeTeam']['name']
        base_data['away_team'] = data['general']['awayTeam']['name']
        votes = data['content']['matchFacts']['poll']['voteResult']['Votes'][0]['Votes']
        base_data['home_probability_win'] = votes[0]
        base_data['draw_probability'] = votes[1]
        base_data['away_probability_win'] = votes[2]
        base_data['home_goals'] = int(data['header']['teams'][0]['score'])
        base_data['away_goals'] = int(data['header']['teams'][1]['score'])
        base_data['ftr'] = 0 if base_data['home_goals'] > base_data['away_goals'] else 1 if base_data['home_goals'] == base_data['away_goals'] else 2
        for information in data['content']['stats']['Periods']['All']['stats']:
            for details in information['stats']:
                if details['title'] == 'Ball possession':
                    home_data['home_ball_possession'] = details['stats'][0]
                    away_data['away_ball_possession'] = details['stats'][1]
                elif details['title'] == "Big chances":
                    home_data['home_big_chances'] = details['stats'][0]
                    away_data['away_big_chances'] = details['stats'][1]
                elif details['title'] == "Big chances missed":
                    home_data['home_big_misses'] = details['stats'][0]
                    away_data['away_big_misses'] = details['stats'][1]
                elif details['title'] == "Fouls committed":
                    home_data['home_fouls_commited'] = details['stats'][0]
                    away_data['away_fouls_commited'] = details['stats'][1]
                elif details['title'] == "Corners":
                    home_data['home_corners'] = details['stats'][0]
                    away_data['away_corners'] = details['stats'][1]
                elif details['title'] == "Shots":
                    home_data['home_shots'] = details['stats'][0]
                    away_data['away_shots'] = details['stats'][1]
                elif details['title'] == "Total shots":
                    home_data['home_total_shots'] = details['stats'][0]
                    away_data['away_total_shots'] = details['stats'][1]
                elif details['title'] == "Shots off target":
                    home_data['home_shots_off_target'] = details['stats'][0]
                    away_data['away_shots_off_target'] = details['stats'][1]
                elif details['title'] == "Shots on target":
                    home_data['home_shots_on_target'] = details['stats'][0]
                    away_data['away_shots_on_target'] = details['stats'][1]
                elif details['title'] == "Blocked shots":
                    home_data['home_blocked_shots'] = details['stats'][0]
                    away_data['away_blocked_shots'] = details['stats'][1]
                elif details['title'] == "Hit woodwork":
                    home_data['home_hit_woodwork'] = details['stats'][0]
                    away_data['away_hit_woodwork'] = details['stats'][1]
                elif details['title'] == "Shots inside box":
                    home_data['home_shots_inside_box'] = details['stats'][0]
                    away_data['away_shots_inside_box'] = details['stats'][1]
                elif details['title'] == "Shots outside box":
                    home_data['home_shots_outside_box'] = details['stats'][0]
                    away_data['away_shots_outside_box'] = details['stats'][1]
                elif details['title'] == "Passes" and details['type'] == 'text':
                    home_data['home_passes'] = details['stats'][0]
                    away_data['away_passes'] = details['stats'][1]
                elif details['title'] == "Accurate passes":
                    values = [int(item.split()[1][1:-2]) for item in details['stats']]
                    home_data['home_accurate_passes'] = values[0]
                    away_data['away_accurate_passes'] = values[1]
                elif details['title'] == "Own half":
                    home_data['home_own_half'] = details['stats'][0]
                    away_data['away_own_half'] = details['stats'][1]
                elif details['title'] == "Opposition half":
                    home_data['home_opposition_half'] = details['stats'][0]
                    away_data['away_opposition_half'] = details['stats'][1]
                elif details['title'] == "Accurate long balls":
                    values = [int(item.split()[1][1:-2]) for item in details['stats']]
                    home_data['home_accurate_long_balls'] = values[0]
                    away_data['away_accurate_long_balls'] = values[1]
                elif details['title'] == "Accurate crosses":
                    values = [int(item.split()[1][1:-2])
                              for item in details['stats']]
                    home_data['home_accurate_crosses'] = values[0]
                    away_data['away_axxurate_crosses'] = values[1]
                elif details['title'] == "Throws":
                    home_data['home_throws'] = details['stats'][0]
                    away_data['away_throws'] = details['stats'][1]
                elif details['title'] == "Offsides":
                    home_data['home_offsides'] = details['stats'][0]
                    away_data['away_offsides'] = details['stats'][1]
                elif details['title'] == "Tackles won":
                    values = [int(item.split()[1][1:-2])
                              for item in details['stats']]
                    home_data['home_tackles_won'] = values[0]
                    away_data['away_tackles_won'] = values[1]
                elif details['title'] == "Interceptions":
                    home_data['home_interceptions'] = details['stats'][0]
                    away_data['away_interceptions'] = details['stats'][1]
                elif details['title'] == "Blocks":
                    home_data['home_blocks'] = details['stats'][0]
                    away_data['away_blocks'] = details['stats'][1]
                elif details['title'] == "Clearances":
                    home_data['home_clearances'] = details['stats'][0]
                    away_data['away_clearances'] = details['stats'][1]
                elif details['title'] == "Keeper saves":
                    home_data['home_keeper_save'] = details['stats'][0]
                    away_data['away_keeper_save'] = details['stats'][1]
                elif details['title'] == "Duels won":
                    home_data['home_duels_won'] = details['stats'][0]
                    away_data['away_duels_won'] = details['stats'][1]
                elif details['title'] == "Ground duels won":
                    values = [int(item.split()[1][1:-2]) for item in details['stats']]
                    home_data['home_geround_duels_won'] = values[0]
                    away_data['away_geround_duels_won'] = values[1]
                elif details['title'] == "Aerial duels won":
                    values = [int(item.split()[1][1:-2])
                              for item in details['stats']]
                    home_data['home_aerial_duels_won'] = values[0]
                    away_data['away_aerial_duels_won'] = values[1]
                elif details['title'] == "Successful dribbles":
                    values = [int(item.split()[1][1:-2])
                              for item in details['stats']]
                    home_data['home_successful_dribbles'] = values[0]
                    away_data['away_successful_dribbles'] = values[1]
                elif details['title'] == "Yellow cards":
                    home_data['home_yellow_cards'] = details['stats'][0]
                    away_data['away_yellow_cards'] = details['stats'][1]
                elif details['title'] == "Red cards":
                    home_data['home_red_cards'] = details['stats'][0]
                    away_data['away_red_cards'] = details['stats'][1]
        each_team = {**base_data, **home_data, **away_data}
        all_teams.append(each_team)
        sleep(15)
    return all_teams                   

In [18]:
single_league = match_details(ids)
df = pd.DataFrame(single_league)
df.head()

KeyboardInterrupt: 